<img style="float: left; margin: 30px 15px 15px 15px;" src="https://oci02.img.iteso.mx/Identidades-De-Instancia/ITESO/Logos%20ITESO/Logo-ITESO-Principal.jpg" width="500" height="250" /> 
    
    
# <font color='navy'> Modelo de Puntuación Crediticia

<font color='black'>

- Ivanna herrera Ibarra
- Ana Sofía Hinojosa Bale
- Luis Fernando Márquez Bañuelos

## <font color='royalblue'> Librerías

In [1]:
import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

## <font color='royalblue'> Funciones

In [2]:
# This is use to fill some parts of the column Num_Credit_Inquiries
def fill_if_neighbors_match(group):
    ffilled = group.ffill()
    bfilled = group.bfill()
    
    # Only fill where both forward and backward give same value
    mask = (group.isna()) & (ffilled == bfilled)
    group[mask] = ffilled[mask]
    
    return group

# This is used to make into numerical values the Credit_History_Age column, which is in the format "X Years and Y Months".
def parse_credit_history_simple(text):
    if pd.isna(text):
        return np.nan
    
    try:
        # Split by "Years and"
        parts = str(text).split('Years and')
        years = int(parts[0].strip())
        
        # Extract months
        months = int(parts[1].replace('Months', '').strip())
        
        return years + (months / 12)
    except:
        return np.nan

# Use to fill the NaN values on the column Credit_History_Age by backfilling and subtracting 1 month for each month back.
def bfill_subtract_monthly(group):
    """
    For leading NaNs, backfill by subtracting 1/12 from the first valid value
    """
    # Find first non-null value
    first_valid_idx = group.first_valid_index()
    
    if first_valid_idx is None:
        return group  # All NaN, can't fill
    
    # Get position of first valid value
    first_value = group.loc[first_valid_idx]
    first_position = group.index.get_loc(first_valid_idx)
    
    # Fill all positions before first valid value
    for i in range(first_position - 1, -1, -1):
        idx = group.index[i]
        if pd.isna(group.loc[idx]):
            months_back = first_position - i
            group.loc[idx] = first_value - (months_back / 12)
    
    return group

## <font color='royalblue'> Datos

In [3]:
data = pd.read_csv('train.csv')
data

/var/folders/60/rl4yk8jj3453bx7hbt074rbc0000gn/T/ipykernel_73651/3529221682.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('train.csv')


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0x25fe9,CUS_0x942c,April,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,34.663572,31 Years and 6 Months,No,35.104023,60.97133255718485,High_spent_Large_value_payments,479.866228,Poor
99996,0x25fea,CUS_0x942c,May,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,_,502.38,40.565631,31 Years and 7 Months,No,35.104023,54.18595028760385,High_spent_Medium_value_payments,496.65161,Poor
99997,0x25feb,CUS_0x942c,June,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,41.255522,31 Years and 8 Months,No,35.104023,24.02847744864441,High_spent_Large_value_payments,516.809083,Poor
99998,0x25fec,CUS_0x942c,July,Nicks,25,078-73-5990,Mechanic,39628.99,3359.415833,4,...,Good,502.38,33.638208,31 Years and 9 Months,No,35.104023,251.67258219721603,Low_spent_Large_value_payments,319.164979,Standard


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

## <font color='royalblue'> Limpieza de Datos

In [5]:
# Safety measure for the dataset
# Note: In this section there will not be data inputation for numeric values, now they are only going to be replaced with NaN, so we can later decide how to impute them.
data = pd.read_csv('train.csv')
data = data.copy()

# This match the Customer_ID with the Name, so we can fill the missing values in the Name column wherever the Customer_ID is the same.
data['Name'] = data.groupby('Customer_ID')['Name'].transform(lambda x: x.ffill().bfill())

/var/folders/60/rl4yk8jj3453bx7hbt074rbc0000gn/T/ipykernel_73651/2769845039.py:3: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('train.csv')


In [6]:
# When trying to convert the Age column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Age'] = data['Age'].str.replace('_', '') # Delete the underscore from the Age column.
data['Age'] = data['Age'].astype(int) # Convert the Age column to numeric. We can do this because we have already removed the underscore from the column.

In [7]:
print(data.Age.min(), data.Age.max()) # Clearly this ages are not correct.
data.loc[data['Age'] > 100, 'Age'] = np.nan # We will replace the ages that are greater than 100 with NaN, because it is not possible for a person to be older than 100 years old.
data.loc[data['Age'] < 0, 'Age'] = np.nan # We will replace the ages that are less than 0 with NaN, because it is not possible for a person to have a negative age.

-500 8698


In [8]:
print(data.Occupation.value_counts()) # There are multiple people with an unknown occupation.
data['Occupation'] = data['Occupation'].replace('_______', 'Unknown')
data = pd.get_dummies(data, columns=['Occupation'])

Occupation
_______          7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64


In [9]:
# When trying to convert the Annual column to numeric, we found that there are some values that contain an underscore.
data['Annual_Income'] = data['Annual_Income'].str.replace('_', '') # Delete the underscore from the Annual_Income column.
data['Annual_Income'] = data['Annual_Income'].astype(float) # Convert the Annual_Income column to numeric.

In [10]:
# For income in hand we use the ffill and bfill method beause if there is a register it is always the same value.
data['Monthly_Inhand_Salary'] = data.groupby('Customer_ID')['Monthly_Inhand_Salary'].transform(lambda x: x.ffill().bfill())

In [11]:
print(data.Num_Bank_Accounts.min(), data.Num_Bank_Accounts.max())
data['Num_Bank_Accounts'] = data['Num_Bank_Accounts'].replace(-1, np.nan) # We will replace the values that are -1 with NaN, because is not possible.
#data.loc[data['Num_Bank_Accounts'] > 15, 'Num_Bank_Accounts'] = np.nan # More than 15 is more than triple the average, so we will replace those values with NaN.

-1 1798


In [12]:
print(data.Num_Credit_Card.min(), data.Num_Credit_Card.max())
#data.loc[data['Num_Credit_Card'] > 21, 'Num_Credit_Card'] = np.nan # More than 21 is more than triple the average, so we will replace those values with NaN.

0 1499


In [13]:
print(data.Interest_Rate.min(), data.Interest_Rate.max())
# We will replace the values that are greater than 900 with NaN.
#data.loc[data['Interest_Rate'] > 900, 'Interest_Rate'] = np.nan

1 5797


In [14]:
# When trying to convert the Num_of_Loan column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Num_of_Loan'] = data['Num_of_Loan'].str.replace('_', '') # Delete the underscore from the Num_of_Loan column.
data['Num_of_Loan'] = data['Num_of_Loan'].astype(int) # Convert the Num_of_Loan column to numeric.

In [15]:
print(data.Num_of_Loan.min(), data.Num_of_Loan.max())
data['Num_of_Loan'] = data['Num_of_Loan'].replace(-100, np.nan) # We will replace the values that are -1 with NaN, because is not possible.
#data.loc[data['Num_of_Loan'] > 25, 'Num_of_Loan'] = np.nan # More tha 25 loans is very unlikely. EXPLAIN THIS.

-100 1496


In [16]:
loan_strings = data['Type_of_Loan'].dropna()

all_loan_types = []

for loan_string in loan_strings:
    loan_string = loan_string.replace(' and ', ', ')
    loans = loan_string.split(',')    
    loans = [loan.strip() for loan in loans if loan.strip()]
    all_loan_types.extend(loans)

unique_loan_types = sorted(set(all_loan_types))

for i, loan in enumerate(unique_loan_types, 1):
    print(f"{loan}")

# Making type of loan dummies
for loan_type in unique_loan_types:
    col_name = f'{loan_type.replace(" ", "_").replace("-", "_")}'
    
    # Check if loan type exists in Type_of_Loan string
    data[col_name] = data['Type_of_Loan'].apply(
        lambda x: True if pd.notna(x) and loan_type in x else False
    )

Auto Loan
Credit-Builder Loan
Debt Consolidation Loan
Home Equity Loan
Mortgage Loan
Not Specified
Payday Loan
Personal Loan
Student Loan


In [17]:
print(data.Delay_from_due_date.min(), data.Delay_from_due_date.max()) # Negative numbers imply early payment given that most cases are Good or Standard.

-5 67


In [18]:
# When trying to convert the Num_of_Delayed_Payment column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Num_of_Delayed_Payment'] = data['Num_of_Delayed_Payment'].str.replace('_', '') # Delete the underscore from the Num_of_Delayed_Payment column.
data['Num_of_Delayed_Payment'] = data['Num_of_Delayed_Payment'].astype(float) # Convert the Num_of_Delayed_Payment column to numeric.
print(data.Num_of_Delayed_Payment.min(), data.Num_of_Delayed_Payment.max())

-3.0 4397.0


In [19]:
# We will replace the values that are less than 0 with NaN, because is not possible to have a negative number of delayed payments.
data.loc[data['Num_of_Delayed_Payment'] < 0, 'Num_of_Delayed_Payment'] = np.nan
# And it does not make sense values as big as 1000.
#data.loc[data['Num_of_Delayed_Payment'] > 252, 'Num_of_Delayed_Payment'] = np.nan 

In [20]:
# When trying to convert the Changed_Credit_Limit column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Changed_Credit_Limit'] = data['Changed_Credit_Limit'].replace('_', np.nan) # Delete the underscore from the Changed_Credit_Limit column.
data['Changed_Credit_Limit'] = data['Changed_Credit_Limit'].astype(float) # Convert the Changed_Credit_Limit column to numeric.

In [21]:
# Filling middle value if neighbors match, becuase the inquiries are historical, so they stay the same or increase.
data['Num_Credit_Inquiries'] = data.groupby('Customer_ID')['Num_Credit_Inquiries'].transform(fill_if_neighbors_match)

In [22]:
# Fill Credit Mix with Unknowns
data['Credit_Mix'] = data['Credit_Mix'].replace('_', 'Unknown')
data = pd.get_dummies(data, columns=['Credit_Mix'])

In [23]:
# When trying to convert the Outstanding_Debt column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Outstanding_Debt'] = data['Outstanding_Debt'].str.replace('_', '') # Delete the underscore from the Outstanding_Debt column.
data['Outstanding_Debt'] = data['Outstanding_Debt'].astype(float) # Convert the Outstanding_Debt column to numeric.

In [24]:
# Make it numerical
data['Credit_History_Age'] = data['Credit_History_Age'].apply(parse_credit_history_simple)

In [25]:
# Filling NaN values in a forward way by customer by adding 1 month.
data['Credit_History_Age'] = data.groupby('Customer_ID')['Credit_History_Age'].transform(
    lambda x: x.interpolate(method='linear')
)

# Filling NaN values in a backward way by customer by subtracting 1 month.
data['Credit_History_Age'] = data.groupby('Customer_ID', group_keys=False)['Credit_History_Age'].apply(
    bfill_subtract_monthly
)

In [26]:
# Make them into dummies
data = pd.get_dummies(data, columns=['Payment_of_Min_Amount'], prefix='Pay_Min')

In [27]:
# When trying to convert the Amount_invested_monthly column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Amount_invested_monthly'] = data['Amount_invested_monthly'].str.replace('_', '') # Delete the underscore from the Amount_invested_monthly column.
data['Amount_invested_monthly'] = data['Amount_invested_monthly'].astype(float) # Convert the Amount_invested_monthly column to numeric.

In [28]:
data['Payment_Behaviour'] = data['Payment_Behaviour'].replace('!@9#%8', 'Unknown') # Replace the error with Unknown in the Payment_Behaviour column.
data = pd.get_dummies(data, columns=['Payment_Behaviour'], prefix='Behaviour') # Make them into dummies

In [29]:
# When trying to convert the Monthly_Balance column to numeric, we found that there are some values that contain an underscore.
# We will replace the underscore with an empty string and then convert the column to numeric.
data['Monthly_Balance'] = data['Monthly_Balance'].str.replace('_', '') # Delete the underscore from the Monthly_Balance column.
data['Monthly_Balance'] = data['Monthly_Balance'].astype(float) # Convert the Monthly_Balance column to numeric.

In [30]:
# All monthly balance lower than 0 are errors, so they are replaced with nan.
data.loc[data['Monthly_Balance'] < 0, 'Monthly_Balance'] = np.nan 

In [31]:
# Select only the numerical columns from the dataset
numerical_cols = data.select_dtypes(include=['float64', 'int64']).columns
X = data[numerical_cols].copy()

# Scaling data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KNN Imputer
# n_neighbors: 10 for large datasets (100K rows)
imputer = KNNImputer(n_neighbors=10, weights='distance')

# Impute missing values
X_imputed_scaled = imputer.fit_transform(X_scaled)

# Convert back to original scale
X_imputed = scaler.inverse_transform(X_imputed_scaled)

# Put back into dataframe
data[numerical_cols] = X_imputed

In [ ]:
#data.to_csv('clean_train.csv', index=False)